In [66]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [67]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

In [68]:
df = df[[
    "Survived", "Pclass", "Sex", "Age",
    "Fare", "SibSp", "Parch", "Embarked", "Name"
]]

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [69]:
df["family_size"] = df["SibSp"] + df["Parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)

In [70]:
df["title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_9435/625526086.py:1: SyntaxWarning: invalid escape sequence '\.'
  df["title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)


In [71]:
# Group rare titles
df["title"] = df["title"].replace([
    "Lady", "Countess", "Capt", "Col", "Don", "Dr",
    "Major", "Rev", "Sir", "Jonkheer", "Dona"
], "Rare")

df["title"] = df["title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

In [72]:
df["age_group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

In [73]:
columns_to_encode = ["Sex", "Embarked", "title", "age_group"]
actual_columns_to_encode = [col for col in columns_to_encode if col in df.columns]

if actual_columns_to_encode:
    df = pd.get_dummies(
        df,
        columns=actual_columns_to_encode,
        drop_first=True
    )
df.drop("Name", axis=1, inplace=True, errors='ignore')

In [74]:
X = df.drop("Survived", axis=1)
X = X.select_dtypes(include=np.number)
y = df["Survived"]

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [76]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

In [77]:
acc = accuracy_score(y_test, pred)

print("IMPROVED MODEL ACCURACY:", acc)

IMPROVED MODEL ACCURACY: 0.8156424581005587
